In [ ]:
!pip install transformers datasets torch pandas evaluate nltk
!pip install rouge-score

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, T5ForConditionalGeneration, AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import evaluate
import os

In [ ]:
# Load datasets
sarcastic_data = pd.read_csv("Final_sarcastic_data.csv")
non_sarcastic_data = pd.read_csv("Final_non_sarcastic_data.csv")

data = pd.merge(sarcastic_data, non_sarcastic_data, on='id')

data['sarcastic'] = data['sarcastic'].astype(str)
data['non_sarcastic'] = data['non_sarcastic'].astype(str)

# Split data (focus on validation dataset which is 20% of total)
train_data, val_test_data = train_test_split(data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(val_test_data, test_size=0.5, random_state=42)

In [ ]:

def format_data(dataframe):
    return {
        'input_text': dataframe['sarcastic'].tolist(),
        'target_text': dataframe['non_sarcastic'].tolist(),
        'id': dataframe['id'].tolist()
    }

train_dataset = format_data(train_data)
val_dataset = format_data(val_data)
test_dataset = format_data(test_data)

# Model and Tokenizer - Changed to t5-small
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [ ]:

def tokenize_data(inputs, targets, tokenizer, max_len=128):
    input_encodings = tokenizer(
        inputs, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt"
    )
    target_encodings = tokenizer(
        targets, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt"
    )
    return input_encodings, target_encodings

train_encodings, train_labels = tokenize_data(
    train_dataset['input_text'], train_dataset['target_text'], tokenizer
)
val_encodings, val_labels = tokenize_data(
    val_dataset['input_text'], val_dataset['target_text'], tokenizer
)


class SarcasmDataset(Dataset):
    def __init__(self, encodings, labels, ids=None, original_targets=None):
        self.input_ids = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels = labels['input_ids']
        self.ids = ids
        self.original_targets = original_targets or [None] * len(self.input_ids)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }
        if self.ids is not None:
            item['id'] = self.ids[idx]
        if self.original_targets[idx] is not None:
            item['original_target'] = self.original_targets[idx]
        return item

train_dataset = SarcasmDataset(train_encodings, train_labels)
val_dataset = SarcasmDataset(
    val_encodings,
    val_labels,
    val_dataset['id'],
    val_dataset['target_text']
)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training (11 epochs as specified)
for epoch in range(11):
    model.train()
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())

# Save the model and tokenizer
os.makedirs("sarcasm_to_non_sarcasm_model", exist_ok=True)
model.save_pretrained("sarcasm_to_non_sarcasm_model")
tokenizer.save_pretrained("sarcasm_to_non_sarcasm_model")
print("Model saved successfully!")

# Evaluation metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def generate_non_sarcastic_texts(val_dataset, tokenizer, model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    generated_texts = []
    true_texts = []

    with torch.no_grad():
        for item in val_dataset:
            input_ids = item['input_ids'].unsqueeze(0).to(device)
            attention_mask = item['attention_mask'].unsqueeze(0).to(device)

            # Generate the model's output
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

            # Decode the generated text
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Get the true text from the original dataset
            true_text = item.get('original_target', tokenizer.decode(item['labels'], skip_special_tokens=True))

            # Append to results
            generated_texts.append(generated_text)
            true_texts.append(true_text)

    # Calculate ROUGE and BLEU scores
    rouge_scores = rouge.compute(predictions=generated_texts, references=true_texts)
    bleu_scores = bleu.compute(predictions=generated_texts, references=true_texts)

    print("ROUGE Scores:", rouge_scores)
    print("BLEU Scores:", bleu_scores)

    # Prepare output with IDs
    output_texts = [
        {
            'id': item.get('id', 'N/A'),
            'non_sarcastic_text': gen_text
        }
        for item, gen_text in zip(val_dataset, generated_texts)
    ]

    return output_texts

# Generate non-sarcastic texts
generated_val_texts = generate_non_sarcastic_texts(val_dataset, tokenizer, model)

# Save to CSV
output_df = pd.DataFrame(generated_val_texts)
output_df.to_csv("generated_non_sarcastic_texts.csv", index=False)
print("Generated texts saved to generated_non_sarcastic_texts.csv")

# Optional: Print a few examples
print("\nSample Generated Non-Sarcastic Texts:")
print(output_df.head())




/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 10: 100%|██████████| 92/92 [00:13<00:00,  6.84it/s, loss=0.211]


Model saved successfully!
ROUGE Scores: {'rouge1': 0.2628214532999427, 'rouge2': 0.14659385864893976, 'rougeL': 0.2540793560923502, 'rougeLsum': 0.25333881505640826}
BLEU Scores: {'bleu': 0.09369588563369748, 'precisions': [0.4613935969868173, 0.22641509433962265, 0.1347517730496454, 0.07588075880758807], 'brevity_penalty': 0.5182743877896598, 'length_ratio': 0.6034090909090909, 'translation_length': 531, 'reference_length': 880}
Generated texts saved to generated_non_sarcastic_texts.csv

Sample Generated Non-Sarcastic Texts:
    id                                 non_sarcastic_text
0  811                  I’ll add 10 meetings to the week.
1  441     I have to overcome another obstacle in my way.
2  894                                                   
3  328                                                   
4   68  My credit card was declined in front of everyone.


In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load the trained model and tokenizer
model_path = "sarcasm_to_non_sarcasm_model"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load the input sarcastic texts
input_df = pd.read_csv("Sarcastic_IAC_data.csv")

# Function to generate non-sarcastic text
def generate_non_sarcastic_text(text, tokenizer, model, max_length=128):
    # Prepare input
    input_ids = tokenizer.encode(
        text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True
    ).to(device)

    # Generate output
    outputs = model.generate(
        input_ids,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    # Decode and return generated text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Generate non-sarcastic texts
results = []
for index, row in input_df.iterrows():
    sarcastic_text = row['text']
    non_sarcastic_text = generate_non_sarcastic_text(sarcastic_text, tokenizer, model)

    results.append({
        'id': row['id'],
        'sarcastic_text': sarcastic_text,
        'non_sarcastic_text': non_sarcastic_text
    })

# Convert to DataFrame and save
output_df = pd.DataFrame(results)
output_df.to_csv("IAC_Output.csv", index=False)

print(f"Generated {len(output_df)} non-sarcastic texts.")
print("Output saved to IAC_Output.csv")

# Optional: Print a few samples
print("\nSample Generated Texts:")
print(output_df.head())

Generated 1630 non-sarcastic texts.
Output saved to IAC_Output.csv

Sample Generated Texts:
   id                                     sarcastic_text  \
0   1  Are you saying kill the chicken by shooting it...   
1   2  It's usually easier to take the immoral path. ...   
2   3  Archie proves, for the umpteenth time, that he...   
3   4  I just bet you thought you were smart just now...   
4   5  I can't believe you think saying "puff piece" ...   

                                  non_sarcastic_text  
0                                                     
1    A deadbeat parent is easier than a good parent.  
2  Archie is utterly incapable of reading for com...  
3  I'll go you several... How about: 1. Murder 2....  
4                                                     
